# 08 — Field inference

**Replication of:** *Presence-absence estimation in audio recordings of tropical frog communities*
(Estrella Terneux, Nicolalde, Nicolalde & Merino-Viteri, arXiv:1901.02495), Section **4.3**

## 1. Objective

This is the stage the article was built for: the frozen segmentation and the frozen detector applied to the pond recordings, which neither saw while they were developed.

**pond sample → 30 s blocks → one band-pass pass per species → candidate intervals → PLP-RASTA of the original signal → frozen mixtures → likelihood ratio → presence or absence per species**

`[PAPER]` Section 4.3 applies the detector to the ten-minute samples of Pond 1 and Pond 2, recorded in five months, and declares a species present in a sample when at least one of its calls is accepted. It then compares the output with ten-variable presence vectors that an expert wrote by hand for **18 samples**.

### What is different here, and what cannot be done

- **The test set is the one the article describes**: the 141 ten-minute samples (22.38 hours), selected as notebook 01 established. An earlier version of this stage processed 546 files, which included short excerpts and files that were not the article's samples.
- **The 18 hand-labelled vectors were not delivered**, so no accuracy, precision or recall can be computed. What the sidecars of the recordings do offer is *partial positive evidence*: some name the species that was heard. That is used, with its limits stated, in notebook 09.
- **`f09` has no model.** It is reported as `NA`, never as absent.
- **Thresholds and segmentation constants are not changed here.** The article tuned its thresholds on annotated field samples; nothing of the kind exists for this detector, and tuning on the unlabelled field would only grade the system on itself.

This notebook produces the tables. Reading them, and testing how much they depend on the choices already made, is notebook 09.

## 2. Setup

**What this section asks, and why it matters.** The scan is heavy: nine band-pass passes over 22 hours of audio and the PLP-RASTA analysis of every candidate. Those parts run in parallel on the CPU. The scoring of the candidates under the nine mixtures runs on the GPU when a usable one
is present and on the CPU otherwise, with the same results; nothing is installed or changed in the environment.

In [1]:
# Every dependency, the display options, the dataset location from .env, and the places the results go.
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from dotenv import load_dotenv
from joblib import Parallel, delayed

from frog_presence import detector as D
from frog_presence import field as FL
from frog_presence import inference as I
from frog_presence.audio import probe_audio_header
from frog_presence.device import array_module, describe
from frog_presence.filenames import classify_env_file, sample_group
from frog_presence.sidecars import read_xmp_date, read_xmp_pond, read_xmp_species

sns.set_theme(style="ticks")
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 140)
pd.set_option("display.max_columns", 80)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models" / "gmm_robust_final"
SEGMENTATION_PATH = PROCESSED_DIR / "candidate_call_segmentation" / "final_segmentation_configuration.json"
RESULTS_DIR = PROCESSED_DIR / "field_presence_absence"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in (MODELS_DIR / "final_detector_manifest.json", SEGMENTATION_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path.name}. Run notebooks 06 and 07 first.")

load_dotenv(REPO_ROOT / ".env")
ponds_value = os.environ.get("PONDS_CALL_DIR")
if not ponds_value or not Path(ponds_value).expanduser().is_dir():
    raise RuntimeError("PONDS_CALL_DIR is not set to an existing folder. Copy .env.example to .env and fill in the paths.")
PONDS_CALL_DIR = Path(ponds_value).expanduser()

XP = array_module()
# Only folder names and relative locations are printed: absolute paths are personal to whoever runs this.
print("Dataset used :", "PONDS_CALL_DIR ->", PONDS_CALL_DIR.name)
print("Results      :", RESULTS_DIR.relative_to(REPO_ROOT).as_posix())
print("CPU workers  :", os.cpu_count(), "| scoring on:", describe(XP))

Dataset used : PONDS_CALL_DIR -> pond-recordings
Results      : data/processed/field_presence_absence
CPU workers  : 32 | scoring on: GPU (cupy, NVIDIA GeForce RTX 4060 Laptop GPU)


## 3. The frozen detector and segmentation

**What this section asks, and why it matters.** Both are loaded exactly as their notebooks froze them, and the detector's model files are checked against the hashes in its manifest, so a changed model cannot go unnoticed.

In [2]:
# Load the frozen detector and the frozen segmentation configuration.
models, thresholds, manifest = D.load_detector(MODELS_DIR)
SEGMENTATION = json.loads(SEGMENTATION_PATH.read_text(encoding="utf-8"))
CODES = list(models)
assert set(CODES) == set(SEGMENTATION["selection"]["C_by_code"])

print("detector status :", manifest["detector_status"], "| feature:", manifest["feature"], "| RASTA start:", manifest["rasta_start"],
      "| Gaussians:", manifest["n_components"])
print("model files checked against their hashes:", len(manifest["gmm_sha256"]))
display(pd.DataFrame({"threshold": thresholds, "C": SEGMENTATION["selection"]["C_by_code"],
                      "padding_ms": SEGMENTATION["selection"]["padding_ms_by_code"],
                      "band_low_hz": {c: b[0] for c, b in SEGMENTATION["paper_frontend"]["species_frequency_bands_hz"].items()},
                      "band_high_hz": {c: b[1] for c, b in SEGMENTATION["paper_frontend"]["species_frequency_bands_hz"].items()}}).round(4))

detector status : frozen | feature: 20_PLP_RASTA | RASTA start: zero | Gaussians: 4
model files checked against their hashes: 9


,threshold,C,padding_ms,band_low_hz,band_high_hz
f01,8.0751,4.00,0,1660.0,3100.0
f02,3.7012,2.00,0,2300.0,3390.0
f03,6.9382,4.00,0,1300.0,1530.0
f04,5.0647,10.00,20,1630.0,3900.0
f05,5.5535,2.50,0,1680.0,3260.0
f06,5.2310,4.00,0,1000.0,2500.0
f07,5.7501,4.00,0,500.0,2720.0
f08,7.1894,1.75,0,800.0,1700.0
f10,9.0146,4.00,0,430.0,3140.0


## 4. The test samples

**What this section asks, and why it matters.** `[PAPER]` The test material is 141 ten-minute samples digitised at 48 kHz. The pond collection holds far more audio than that: full cassette sides, the ten-minute samples, short excerpts cut from them, and recordings from another device.
Which files are the article's samples is decided by the filename (`frog_presence.filenames`), and notebook 01 showed that the ten-minute samples are the files named `ccNNNx_SPn`. The selection must reproduce what notebook 01 found: 141 files, 22.38 hours, lossless.

In [3]:
# Probe every audio file of the pond collection, assign roles from the filenames, and select the ten-minute samples.
audio_paths = sorted(p for p in PONDS_CALL_DIR.rglob("*") if p.is_file() and not p.name.startswith("._")
                     and (p.suffix.lower() in (".wav", ".mp3") or p.name.lower().endswith("wav")))
inventory = pd.DataFrame([{"path": p, "rel": p.relative_to(PONDS_CALL_DIR).as_posix(), "name": p.name, "stem": p.stem.lower(),
                           "role": classify_env_file(p.name), **{k: v for k, v in probe_audio_header(p).items() if k in ("subtype", "sample_rate", "duration_s")}}
                          for p in audio_paths])

by_role = inventory.groupby("role").agg(files=("name", "size"), hours=("duration_s", lambda s: s.sum() / 3600), median_s=("duration_s", "median"),
                                        encodings=("subtype", lambda s: "/".join(sorted(s.dropna().unique()))))
display(by_role.round(2).sort_values("files", ascending=False))

samples = inventory[(inventory.role == "SP base") & (inventory.subtype != "GSM610")].copy().reset_index(drop=True)
samples["sample_id"] = samples.rel.str.replace("/", "__", regex=False).str.replace(r"\.wav$", "", regex=True, case=False)
samples["group"] = samples.name.map(sample_group)
print("audio files in the collection        :", len(inventory))
print("test samples (SP base, lossless)     :", len(samples), "| hours:", round(samples.duration_s.sum() / 3600, 2))
print("sample rates / encodings             :", sorted(samples.sample_rate.astype(int).unique()), "/", sorted(samples.subtype.unique()))
print("duration quantiles (s) 0/5/50/95/100 :", samples.duration_s.quantile([0, .05, .5, .95, 1]).round(1).tolist())
assert samples.sample_id.is_unique and samples.group.notna().all()

,files,hours,median_s,encodings
role,,,,
SP sub,475,5.90,6.34,FLOAT/GSM610/PCM_16
SP base,262,42.03,581.99,GSM610/PCM_16
side master,88,41.89,1753.26,GSM610/PCM_16
LS recorder,19,10.69,1840.06,PCM_16/PCM_24
unclassified,2,0.17,308.32,GSM610


audio files in the collection        : 846
test samples (SP base, lossless)     : 141 | hours: 22.38
sample rates / encodings             : [np.int64(48000)] / ['PCM_16']
duration quantiles (s) 0/5/50/95/100 : [122.7, 523.5, 581.0, 605.6, 626.3]


`[OBSERVED]` The pond collection holds **846 audio files** of five kinds. The ten-minute samples are the `SP base` files (262 files, 42.03 hours), of which 141 are lossless; the others are GSM 6.10, which cannot be undone.
The selection gives **141 files, 22.38 hours, all PCM_16 at 48000 Hz**, the figures notebook 01 found for the article's material.

`[OBSERVED]` The samples are not exactly ten minutes: the median is 581.0 seconds, the 5th percentile 523.5 and the range 122.7 to 626.3 seconds. `[PAPER]` The article's 23.5 hours is the nominal 141 × 10 minutes; the measured duration is 22.38 hours.

## 5. Pond, date and the evidence the sidecars hold

**What this section asks, and why it matters.** Each recording has an `.xmp` sidecar written by the cataloguer. For these samples it states the **pond** (in `Location`) and the **date** (in `Fecha`), which gives the temporal structure the article analyses. Some sidecars also name the **species heard** (`Especie`), either on the
sample itself or on the excerpts cut from it.

`[PAPER]` The recordings were made on 8 days in February, 17 in April, 12 in July, 16 in August and 13 in September, at Pond 1 around 21:00 and Pond 2 around 02:00, one ten-minute sample a day.

An unnamed species is **not** an absent one: the sidecars were written to catalogue excerpts, not to list everything that is audible. They are positive evidence only.

In [4]:
# Read pond and date from each sample's sidecar, and gather the species named on the sample and on the excerpts cut from it.
sidecars = {p.stem.lower(): p for p in PONDS_CALL_DIR.rglob("*.xmp") if not p.name.startswith("._")}
samples["pond"] = [read_xmp_pond(sidecars[s]) if s in sidecars else None for s in samples.stem]
samples["date"] = [read_xmp_date(sidecars[s]) if s in sidecars else None for s in samples.stem]
samples["month"] = [d.strftime("%B") if d else None for d in samples.date]
print("samples with a sidecar:", int(samples.stem.isin(sidecars).sum()), "of", len(samples))
print("samples with a pond   :", int(samples.pond.notna().sum()), "| with a date:", int(samples.date.notna().sum()))
display(samples.groupby(["month", "pond"], dropna=False).size().unstack("pond", fill_value=0))

months = samples.dropna(subset=["date"]).groupby("month").date.nunique()
PAPER_DAYS = {"February": 8, "April": 17, "July": 12, "August": 16, "September": 13}
display(pd.DataFrame({"distinct_dates_here": months, "days_in_paper": pd.Series(PAPER_DAYS)}).reindex(list(PAPER_DAYS) + [m for m in months.index if m not in PAPER_DAYS]))

samples with a sidecar: 141 of 141
samples with a pond   : 139 | with a date: 139


pond,P1,P2,NaN
month,,,
April,18,17,0
August,16,18,0
February,9,8,0
July,14,12,0
September,14,13,0
NaN,0,0,2


,distinct_dates_here,days_in_paper
February,10,8
April,19,17
July,16,12
August,18,16
September,16,13


In [5]:
# Map the species named in the sidecars to the species codes, and attach to each sample what its own sidecar and its excerpts name.
EPITHET_TO_CODE = {"alfaroi": "f01", "bifurcus": "f02", "bifircus": "f02", "cinerascens": "f03", "conspicillatus": "f04",
                   "discodactylus": "f05", "fuscifacies": "f06", "fuscufaceus": "f06", "lanciformis": "f07",
                   "margaritifer": "f08", "margaritifera": "f08", "parviceps": "f09", "petersi": "f10"}


def codes_in(name: str) -> set:
    return {code_ for epithet, code_ in EPITHET_TO_CODE.items() if epithet in name}


named_by_sidecar = {stem: read_xmp_species(path) for stem, path in sidecars.items()}
all_names = pd.Series([n for names in named_by_sidecar.values() for n in names]).value_counts()
mapping = pd.DataFrame({"sidecars": all_names, "code": [",".join(sorted(codes_in(n))) or "unmapped" for n in all_names.index]})
display(mapping)

group_of = {stem: sample_group(stem + ".wav") for stem in sidecars}
evidence_by_group = {}
for stem, names in named_by_sidecar.items():
    if group_of[stem]:
        evidence_by_group.setdefault(group_of[stem], set()).update(c for n in names for c in codes_in(n))
samples["named_species"] = [sorted(evidence_by_group.get(g, set())) for g in samples.group]
samples["named_modelled"] = [[c for c in cs if c in models] for cs in samples.named_species]

print("samples whose sidecar or excerpts name at least one species :", int(samples.named_species.map(len).gt(0).sum()), "of", len(samples))
print("... at least one modelled species                          :", int(samples.named_modelled.map(len).gt(0).sum()))
print("names that map to no code                                  :", int((mapping.code == "unmapped").sum()))
display(pd.Series([c for cs in samples.named_species for c in cs]).value_counts().rename("samples naming the species").to_frame())

,sidecars,code
bifurcus,70,f02
cinerascens,49,f03
petersi,5,f10
fuscifacies,1,f06
boans,1,unmapped
pentadactylus,1,unmapped
planiceps,1,unmapped
deridens,1,unmapped


samples whose sidecar or excerpts name at least one species : 89 of 141
... at least one modelled species                          : 89
names that map to no code                                  : 4


,samples naming the species
f02,52
f03,40
f10,4
f06,1


`[OBSERVED]` All 141 samples have a sidecar, and **139** give a pond and a date. The samples fall in February (9 from Pond 1 and 8 from Pond 2), April (18 and 17), July (14 and 12), August (16 and 18) and September (14 and 13). In every month the sidecars hold **more distinct dates than the article's number of days**
(10 against 8 in February, 19 against 17 in April, 16 against 12 in July, 18 against 16 in August, 16 against 13 in September).

`[OBSERVED]` The sidecars name a species for **89 of the 141 samples**, on the sample or on an excerpt cut from it, and the species are few: `f02` in 52 samples, `f03` in 40, `f10` in 4 and `f06` in 1. Four names map to no modelled species (*boans*, *pentadactylus*, *planiceps*, *deridens*). **No sample names `f01`, `f04`, `f05`, `f07` or `f08`.**

`[INFERENCE]` The sidecars give positive evidence for two species with some weight (`f02`, `f03`) and almost none for the rest, so they can test the detector for those two and say little about the others. A species that is not named is not absent. The difference in the number of dates is not explained by the data: the sidecar dates may include
samples of adjacent days that the article's schedule does not list.

## 6. Scan the samples

**What this section asks, and why it matters.** This is the long step. For each sample, in parallel: the recording is read, segmented once per species band with that species' `C` and padding, and the PLP-RASTA frames of every candidate are computed from the original signal. In the main process the candidates are scored under the nine
mixtures, identified with the highest mean frame log-likelihood, and verified against the species' threshold.

`[PAPER]` Candidates of different bands are kept apart (the same event can be proposed by several bands), and each is classified whatever band proposed it. A candidate shorter than one 20 ms analysis frame has no frames and cannot be scored; it is recorded as invalid and never accepted.

In [6]:
# Scan every test sample: segment and extract frames in parallel, score and decide in the main process.
# Half of the logical cores: the work is memory-hungry (a ten-minute recording is read in full by each worker).
N_WORKERS = max(1, (os.cpu_count() or 2) // 2)


def scan_recording(path, sample_id):
    """Segment one recording and extract the frames of its candidates."""
    started = time.time()
    audio, sr = sf.read(path, dtype="float64")
    audio = audio.mean(axis=1) if audio.ndim == 2 else audio
    intervals = FL.segment_recording(audio, sr, SEGMENTATION)
    frames, lengths = FL.candidate_frames(audio, sr, intervals)
    return sample_id, sr, len(audio), intervals, frames, lengths, time.time() - started


started = time.time()
tables, log_rows = [], []
stream = Parallel(n_jobs=N_WORKERS, return_as="generator")(delayed(scan_recording)(r.path, r.sample_id) for r in samples.itertuples())
for done, (sample_id, sr, n_samples, intervals, frames, lengths, seconds) in enumerate(stream, start=1):
    out = I.decide_scores(models, thresholds, I.mean_log_likelihoods_stacked(models, frames, lengths, XP))
    hypothesis = np.array([CODES[h] if h >= 0 else None for h in out["hypothesis"]], dtype=object)
    table = pd.DataFrame({
        "sample_id": sample_id, "band": intervals.band.to_numpy(), "start_s": intervals.start.to_numpy() / sr,
        "end_s": intervals.end.to_numpy() / sr, "duration_s": (intervals.end - intervals.start).to_numpy() / sr,
        "n_frames": lengths, "valid": out["valid"], "hypothesis": hypothesis, "llr": out["llr"],
        "threshold": [thresholds[h] if h else np.nan for h in hypothesis], "accepted": out["accepted"]})
    tables.append(table)
    log_rows.append({"sample_id": sample_id, "sample_rate": sr, "duration_s": n_samples / sr, "candidates": len(table),
                     "valid": int(out["valid"].sum()), "accepted": int(out["accepted"].sum()), "worker_seconds": seconds})
    if done % 20 == 0 or done == len(samples):
        print(f"  {done:>3}/{len(samples)} samples | {sum(len(t) for t in tables):>8} candidates | {time.time() - started:6.0f} s")

candidates = pd.concat(tables, ignore_index=True)
run_log = pd.DataFrame(log_rows)
print("\nscan finished in", round(time.time() - started), "s on", describe(XP))
print("samples:", len(run_log), "| candidates:", len(candidates), "| audio scanned (hours):", round(run_log.duration_s.sum() / 3600, 2))

   20/141 samples |   105977 candidates |    230 s


   40/141 samples |   220469 candidates |    342 s


   60/141 samples |   301843 candidates |    414 s


   80/141 samples |   361413 candidates |    514 s


  100/141 samples |   427342 candidates |    642 s


  120/141 samples |   505931 candidates |    741 s


  140/141 samples |   577873 candidates |    800 s
  141/141 samples |   579540 candidates |    800 s

scan finished in 800 s on GPU (cupy, NVIDIA GeForce RTX 4060 Laptop GPU)
samples: 141 | candidates: 579540 | audio scanned (hours): 22.38


`[OBSERVED]` The scan covers **141 samples and 22.38 hours** and produces **579,540 candidates**, in 800 seconds with the scoring done on the GPU. Every candidate has at least one analysis frame, so all of them could be scored.

## 7. Presence and absence

**What this section asks, and why it matters.** `[PAPER]` A species is present in a sample when at least one of its calls is accepted: a candidate whose winning hypothesis is that species and whose likelihood ratio reaches its threshold. One accepted candidate is enough, which makes the presence vector very sensitive to the false
alarms of a single species.

`f09` is `NA`: there is no model, so no statement about it is possible. `NA` is a limit of the replication, not ecological absence.

In [7]:
# Aggregate the accepted candidates into one presence vector per sample, with f09 as NA.
accepted = candidates[candidates.accepted]
present = accepted.groupby(["sample_id", "hypothesis"]).size().unstack(fill_value=0).reindex(index=samples.sample_id, columns=CODES, fill_value=0) > 0
presence = present.astype(int)
presence.insert(8, "f09", np.nan)
presence = presence[[f"f{i:02d}" for i in range(1, 11)]].rename(columns=lambda c: f"presence_{c}")
presence = samples[["sample_id", "pond", "date", "month", "duration_s"]].merge(presence.reset_index(), on="sample_id")

summary = pd.DataFrame({"modelled": [c in models for c in [f"f{i:02d}" for i in range(1, 11)]],
                        "samples_present": [presence[f"presence_{c}"].sum() if c in models else np.nan for c in [f"f{i:02d}" for i in range(1, 11)]]},
                       index=[f"f{i:02d}" for i in range(1, 11)])
summary["share_of_samples"] = summary.samples_present / len(presence)
display(summary.round(3))
print("samples:", len(presence), "| samples with no accepted detection at all:", int((presence.filter(like="presence_f").fillna(0).sum(axis=1) == 0).sum()))
print("samples with all nine modelled species present:", int((presence[[f"presence_{c}" for c in CODES]].sum(axis=1) == len(CODES)).sum()))

,modelled,samples_present,share_of_samples
f01,True,18.0,0.128
f02,True,141.0,1.000
f03,True,100.0,0.709
f04,True,90.0,0.638
f05,True,57.0,0.404
f06,True,120.0,0.851
f07,True,62.0,0.440
f08,True,44.0,0.312
f09,False,NaN,NaN
f10,True,30.0,0.213


samples: 141 | samples with no accepted detection at all: 0
samples with all nine modelled species present: 0


`[OBSERVED]` The presence output is saturated. **`f02` is reported present in all 141 samples (1.000)**, `f06` in 120 (0.851), `f03` in 100 (0.709) and `f04` in 90 (0.638), then `f07` in 62 (0.440), `f05` in 57 (0.404), `f08` in 44 (0.312), `f10` in 30 (0.213) and `f01` in 18 (0.128). No sample has no accepted detection, and none has all nine species.

`[INFERENCE]` A species that is present in every sample carries no information about the samples. With one accepted candidate enough to declare presence, a few false alarms in ten minutes of pond audio are enough for the result to saturate, and that is what the shares of `f02`, `f06`, `f03` and `f04` look like.
`[PAPER]` The article reports that `f01`, `f04`, `f08` and `f09` were absent from the February samples that were labelled by hand, whereas this output reports `f04` in most samples. Whether that holds for February specifically is examined in notebook 09.

## 8. First look at the detections

**What this section asks, and why it matters.** Before the tables are saved, a few counts that show whether the output is plausible at all: how many candidates the front end proposes per hour, what share the detector accepts, and which species it assigns them to. The deeper examination, and how much each result depends on the thresholds and segmentation, is in
notebook 09.

In [8]:
# Candidates per hour, the share accepted, and the species that the winning hypotheses and accepted detections go to.
hours = run_log.duration_s.sum() / 3600
print(f"candidates per hour of audio       : {len(candidates) / hours:,.0f}")
print(f"valid (at least one frame)         : {candidates.valid.mean():.3f}")
print(f"accepted, of all candidates        : {candidates.accepted.mean():.3f}")
print(f"median candidate duration (s)      : {candidates.duration_s.median():.3f}")

by_hypothesis = candidates[candidates.valid].groupby("hypothesis").agg(candidates=("llr", "size"), accepted=("accepted", "sum"),
                                                                     acceptance=("accepted", "mean"), median_llr=("llr", "median"))
by_hypothesis["share_of_valid"] = by_hypothesis.candidates / by_hypothesis.candidates.sum()
by_hypothesis["share_of_accepted"] = by_hypothesis.accepted / by_hypothesis.accepted.sum()
display(by_hypothesis.round(3).sort_values("accepted", ascending=False))

print("accepted candidates by the band that proposed them (rows) and the species they were assigned to (columns):")
display(pd.crosstab(accepted.band, accepted.hypothesis))

candidates per hour of audio       : 25,899
valid (at least one frame)         : 1.000
accepted, of all candidates        : 0.610
median candidate duration (s)      : 0.180


,candidates,accepted,acceptance,median_llr,share_of_valid,share_of_accepted
hypothesis,,,,,,
f02,275884,250814,0.909,6.947,0.476,0.710
f03,195327,82808,0.424,6.350,0.337,0.234
f06,61811,10842,0.175,3.834,0.107,0.031
f05,5482,3020,0.551,5.931,0.009,0.009
f04,13195,2645,0.200,3.738,0.023,0.007
f07,6267,1382,0.221,3.654,0.011,0.004
f01,1354,941,0.695,10.122,0.002,0.003
f10,18748,574,0.031,3.881,0.032,0.002
f08,1472,406,0.276,5.140,0.003,0.001


accepted candidates by the band that proposed them (rows) and the species they were assigned to (columns):


hypothesis,f01,f02,f03,f04,f05,f06,f07,f08,f10
band,,,,,,,,,
f01,77,33817,9739,360,367,1802,157,71,34
f02,254,32537,8049,132,35,1654,10,40,15
f03,54,16018,8178,500,89,582,140,26,88
f04,6,27585,4373,410,1200,1432,91,13,13
f05,251,33886,9241,161,112,1740,149,66,24
f06,41,28584,9313,378,519,857,148,24,51
f07,59,33392,10071,338,456,1078,255,32,154
f08,129,10511,13235,88,14,105,147,73,58
f10,70,34484,10609,278,228,1592,285,61,137


`[OBSERVED]` The front end proposes **25,899 candidates per hour** of audio, with a median duration of 0.180 seconds, and the detector accepts **0.610** of them.

`[OBSERVED]` The winning hypothesis is `f02` for **47.6 %** of the candidates, and 0.909 of those are accepted; `f03` wins 33.7 %, with 0.424 accepted; `f06` wins 10.7 %, with 0.175 accepted. Together `f02`, `f03` and `f06` account for **97.5 %** of all accepted candidates (0.710, 0.234 and 0.031),
and the remaining six species for 2.5 %.

`[OBSERVED]` The same species dominate whichever band proposed the candidate: of the accepted candidates from the `f08` band, 10,511 go to `f02` and 13,235 to `f03`, and only 73 to `f08`; from the `f04` band 27,585 go to `f02` and 410 to `f04`.

`[INFERENCE]` The band does not decide the outcome, so this is not a band effect. It is the behaviour expected of a closed set of mixtures that has no model of anything but nine frog species: a segment of noise is assigned to whichever model fits it best, and for a large share of the candidates that is `f02` or `f03`, with a ratio above the
threshold set at the weakest calibration call. How much of the output that accounts for is examined in notebook 09, by varying the thresholds and the shortest candidate counted, without choosing either.

## 9. Save the tables

**What this section asks, and why it matters.** The tables are the input of notebook 09, so they are written with the information needed to interpret them: the metadata of each sample (pond, date, the species its sidecars name), every candidate with its decision, and a manifest tying the run to the detector and
segmentation configuration that produced it.

In [9]:
# Write the sample table, the presence table, every candidate, the run log and a manifest, and list only what was written.
sample_table = samples.drop(columns=["path"]).assign(named_species=lambda t: t.named_species.map(",".join),
                                                    named_modelled=lambda t: t.named_modelled.map(",".join))
sample_table.to_csv(RESULTS_DIR / "samples.csv", index=False)
presence.to_csv(RESULTS_DIR / "presence.csv", index=False)
candidates.to_csv(RESULTS_DIR / "candidates.csv.gz", index=False)
run_log.to_csv(RESULTS_DIR / "run_log.csv", index=False)

run_manifest = {
    "notebook": "08_field_inference",
    "test_samples": int(len(samples)), "hours": float(samples.duration_s.sum() / 3600),
    "selection": "files named ccNNNx_SPn (role 'SP base') that are not GSM 6.10",
    "detector": {"status": manifest["detector_status"], "feature": manifest["feature"], "rasta_start": manifest["rasta_start"],
                 "n_components": manifest["n_components"], "gmm_sha256": manifest["gmm_sha256"], "thresholds": {c: float(v) for c, v in thresholds.items()}},
    "segmentation": SEGMENTATION["selection"],
    "candidates": int(len(candidates)), "accepted_candidates": int(candidates.accepted.sum()),
    "thresholds_retuned_on_ponds": False, "segmentation_retuned_on_ponds": False,
    "field_metrics_computed": False,
    "field_metric_reason": "the complete ten-variable vectors of the article's 18 labelled samples were not delivered",
    "f09": "NA: no model",
}
(RESULTS_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

print("written to", RESULTS_DIR.relative_to(REPO_ROOT).as_posix())
for name in ("samples.csv", "presence.csv", "candidates.csv.gz", "run_log.csv", "run_manifest.json"):
    print(f"  {name:<22} {(RESULTS_DIR / name).stat().st_size / 1e6:>8.2f} MB")

written to data/processed/field_presence_absence
  samples.csv                0.02 MB
  presence.csv               0.01 MB
  candidates.csv.gz         11.29 MB
  run_log.csv                0.01 MB
  run_manifest.json          0.00 MB


## 10. Conclusions and limitations

**What this notebook delivers.** The article's field stage, run on its own test material: the 141 ten-minute samples (22.38 hours), segmented with the frozen front end and classified with the frozen detector, with the pond, the date and the species named in the sidecars recovered for each sample. The tables
(candidates with their decisions, the presence vectors, the sample metadata and a manifest tying the run to the detector and segmentation that made it) are in `data/processed/field_presence_absence`.

**What was found.**

1. **The test set is the article's**: 141 files, 22.38 hours, lossless, found by their names. An earlier version of this stage processed 546 files that included short excerpts and other material.
2. **The front end proposes about 26,000 candidates per hour**, and the detector accepts 61 % of them.
3. **The presence output is saturated**: `f02` is present in every sample and `f06`, `f03` and `f04` in 85 %, 71 % and 64 %. Three species account for 97.5 % of the accepted candidates.
4. **The sidecars name species for 89 samples, but only four species** (`f02`, `f03`, `f10`, `f06`), so they can test two of the nine.

**Decisions taken here, which the article does not state.**

1. **The test samples are those named `ccNNNx_SPn` that are lossless.**
2. **Each species band is segmented separately** and every candidate is classified whatever band proposed it, with candidates of the same band merged.
3. **Scoring runs on the GPU when one is usable** and on the CPU otherwise; the results do not depend on it.
4. **Nothing is retuned on the ponds.**

**Limitations.**

- **No field labels.** The 18 hand-labelled vectors were not delivered, so no accuracy, recall or precision can be computed, and `f09` is `NA`.
- **The detector has no model of anything but nine frog species**, and was calibrated against frog calls only, so it cannot reject what is not a frog call by design.
- **One accepted candidate declares presence**, which makes the vectors as sensitive as the worst false-alarm rate of each species.
- **The sidecars are positive evidence only** and cover four species.

**What the next stage inherits.** A presence table that saturates for the species the detector tends to assign noise to. Reading it requires asking how it changes with the choices made without field labels, which is what notebook 09 does.